# Amazon ML Challenge 2026 — Data Analysis and Normalization

**Member 2 Deliverable — Exploratory Data Analysis & Preprocessing**

---

## 1. Objective

This notebook profiles **Source 1**, **Source 2**, **Source 3**, and the **ground-truth** file for the Amazon ML Challenge 2026 entity-matching task.

Goals:
- Understand the structure, size, and field types of each dataset.
- Identify data-quality issues: missing values, empty strings, duplicate entity IDs.
- Characterise country distributions and name/address length distributions.
- Analyse the ground-truth match cardinality (zero / one / multiple matches per S1 entity).
- Study real noise patterns in business names and addresses.
- Design and demonstrate a normalization strategy that reduces surface noise without discarding meaningful information.

**Scope:** This notebook covers EDA and normalization only. Blocking, candidate generation, fuzzy matching, model training, and final prediction are handled in later stages by other team members.

---

## 2. Dataset Loading

The four training files are **tab-separated values (TSV)** files.

| File | Role |
|---|---|
| `train_source1.tsv` | Primary (left-hand side) entities |
| `train_source2.tsv` | Secondary source to match against S1 |
| `train_source3.tsv` | Secondary source to match against S1 |
| `train_ground_truth.tsv` | Known correct matches from S1 to S2/S3 |

In [3]:
import pandas as pd

s1 = pd.read_csv("/content/train_source1.tsv", sep="\t")
s2 = pd.read_csv("/content/train_source2.tsv", sep="\t")
s3 = pd.read_csv("/content/train_source3.tsv", sep="\t")
gt = pd.read_csv("/content/train_ground_truth.tsv", sep="\t")

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)
print("GT:", gt.shape)

S1: (99098, 4)
S2: (108019, 4)
S3: (120884, 4)
GT: (72960, 2)


---

## 3. Dataset Overview

Inspect column names, data types, and the first few rows of each source.

In [4]:
print("S1 columns:", s1.columns.tolist())
print("S2 columns:", s2.columns.tolist())
print("S3 columns:", s3.columns.tolist())
print("GT columns:", gt.columns.tolist())

S1 columns: ['entity_id', 'business_name', 'business_address', 'country']
S2 columns: ['entity_id', 'business_name', 'business_address', 'country']
S3 columns: ['entity_id', 'business_name', 'business_address', 'country']
GT columns: ['source1_entity_id', 'matched_entity_ids']


In [5]:
display(s1.head())
display(s2.head())
display(s3.head())
display(gt.head())

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


---

## 4. Data Quality Analysis

For each source we examine:
- **Missing values** (`NaN`) per column
- **Duplicate entity IDs** (should be zero for a clean source)
- **Empty business names / addresses** (non-null but blank strings)
- **Country distribution** (value counts, including NaN)

> ⚠️ Country is treated as a free-form open-set string. The pipeline does **not** assume a fixed list of countries.

In [ ]:
# ==============================
# STEP 2: DATA QUALITY ANALYSIS
# ==============================

datasets = {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3
}

for name, df in datasets.items():
    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicate entity IDs:")
    print(df["entity_id"].duplicated().sum())

    print("\nCountries:")
    print(df["country"].value_counts(dropna=False))

    print("\nEmpty business names:",
          df["business_name"].fillna("").str.strip().eq("").sum())

    print("Empty addresses:",
          df["business_address"].fillna("").str.strip().eq("").sum())

---

## 5. Country Distribution

The `country` field is a free-form string that may contain any country or region. The distribution below reflects what actually appears in the data and must **not** be used to hard-code a fixed country list in any downstream stage.

In [8]:
for name, df in datasets.items():
    print("\n", name)
    print(df["country"].value_counts())


 Source 1
country
US       59338
India    39759
Name: count, dtype: int64

 Source 2
country
US       64737
India    43281
Name: count, dtype: int64

 Source 3
country
US       72104
India    48779
Name: count, dtype: int64


---

## 6. Name and Address Length Analysis

Descriptive statistics (count, mean, std, min, max, quartiles) for character-level lengths of raw `business_name` and `business_address` fields across all three sources.

High variance in lengths is a common indicator of abbreviation and formatting noise.

In [7]:
for name, df in datasets.items():

    name_len = df["business_name"].fillna("").astype(str).str.len()
    addr_len = df["business_address"].fillna("").astype(str).str.len()

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("\nBusiness name length:")
    print(name_len.describe())

    print("\nAddress length:")
    print(addr_len.describe())


Source 1

Business name length:
count    99098.000000
mean        24.041918
std          7.730900
min          3.000000
25%         18.000000
50%         24.000000
75%         30.000000
max         71.000000
Name: business_name, dtype: float64

Address length:
count    99098.000000
mean        52.092726
std         25.368050
min         13.000000
25%         33.000000
50%         41.000000
75%         70.000000
max        222.000000
Name: business_address, dtype: float64

Source 2

Business name length:
count    108019.000000
mean         25.081180
std           8.889834
min           2.000000
25%          18.000000
50%          25.000000
75%          31.000000
max         104.000000
Name: business_name, dtype: float64

Address length:
count    108019.000000
mean         46.238865
std          24.798145
min           0.000000
25%          30.000000
50%          37.000000
75%          62.000000
max         204.000000
Name: business_address, dtype: float64

Source 3

Business name lengt

---

## 7. Ground Truth Analysis

The ground-truth file maps each Source 1 entity to zero, one, or multiple matching entities from Source 2 and/or Source 3.

- **Zero matches** — the S1 entity has no corresponding entity in S2/S3.
- **One match** — the S1 entity matches exactly one entity in S2 or S3.
- **Multiple matches** — the S1 entity matches two or more entities (e.g. one in S2 and one in S3).

Understanding this distribution is essential for selecting the right objective function and evaluation metric in the matching stage.

In [9]:
print("Ground truth rows:", len(gt))

print("\nMissing values:")
print(gt.isna().sum())

print("\nFirst 20 ground-truth records:")
display(gt.head(20))

Ground truth rows: 72960

Missing values:
source1_entity_id        0
matched_entity_ids    4090
dtype: int64

First 20 ground-truth records:


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [10]:
print(gt["matched_entity_ids"].head(20).tolist())

['S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364', 'S2-249013014,S2-197070651,S3-478195123,S3-384364074', 'S2-790675320,S2-479876582,S3-878454467', 'S2-153058913,S2-24659151,S3-679606215', 'S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785', 'S2-755677256,S3-187831601,S3-641489370,S3-476250621', 'S2-660036492,S3-804600254', 'S3-274817120,S3-312496301', 'S2-648035184,S3-588502663', 'S2-383871912,S3-74481402,S3-576451439', 'S2-356983532,S3-352439310', 'S2-7028416,S2-442723188,S2-157073701,S3-523120965', 'S2-487600131,S2-582477216,S2-392804085,S3-200008747,S3-729771680,S3-249331830', 'S2-736616474,S2-680265918,S2-51486805,S3-925631694,S3-461175723,S3-850112871', 'S2-120366543,S2-939389287,S3-96572514', 'S2-994658326,S2-235117490,S2-353308450,S2-173295926,S3-858763214,S3-33665521,S3-555791452', 'S2-580419223,S3-164220452', 'S2-483615364,S3-619529814', 'S2-836452886,S3-413669121', 'S2-938895481,S3-138350041']


In [11]:
match_count = (
    gt["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .str.strip()
    .apply(lambda x: 0 if x == "" else len(x.split(",")))
)

print("0 matches :", (match_count == 0).sum())
print("1 match   :", (match_count == 1).sum())
print("2+ matches:", (match_count >= 2).sum())

print("\nMatch count distribution:")
print(match_count.value_counts().sort_index())

0 matches : 4090
1 match   : 3921
2+ matches: 64949

Match count distribution:
matched_entity_ids
0      4090
1      3921
2     12364
3     17600
4     16183
5     10580
6      5412
7      2076
8       579
9       136
10       16
11        3
Name: count, dtype: int64


In [12]:
for i in range(min(10, len(gt))):
    s1_id = gt.iloc[i]["source1_entity_id"]
    matches = gt.iloc[i]["matched_entity_ids"]

    print("\nS1:", s1_id)
    print("Matches:", matches)

    row = s1[s1["entity_id"] == s1_id]

    if len(row):
        print("S1 record:")
        display(row)


S1: S1-965667
Matches: S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364

S1: S1-55344266
Matches: S2-249013014,S2-197070651,S3-478195123,S3-384364074

S1: S1-343815751
Matches: S2-790675320,S2-479876582,S3-878454467

S1: S1-656753428
Matches: S2-153058913,S2-24659151,S3-679606215

S1: S1-102811957
Matches: S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785

S1: S1-18727616
Matches: S2-755677256,S3-187831601,S3-641489370,S3-476250621

S1: S1-318373630
Matches: S2-660036492,S3-804600254

S1: S1-86989137
Matches: S3-274817120,S3-312496301

S1: S1-29845983
Matches: S2-648035184,S3-588502663

S1: S1-789009573
Matches: S2-383871912,S3-74481402,S3-576451439


---

## 8. Real Data Noise Examples

The cells below display real records from the dataset where normalization changes the raw value. These illustrate the main noise patterns encountered:

- **Punctuation variation** — periods, commas, hyphens in names and addresses
- **Capitalization variation** — mixed case, all-caps, sentence case
- **Abbreviation variation** — `'St.'` vs `'Street'`, `'Co.'` vs `'Company'`
- **Address formatting variation** — different ordering of street/city/zip
- **Unicode compatibility variations** — NFKC normalization resolves fullwidth forms and combining characters; it does not transliterate accented characters

---

## 9. Normalization

### Strategy

The normalization pipeline (also implemented in `src/preprocess.py`) applies these steps:

| Step | Description |
|---|---|
| 1. Null handling | Return `""` for `NaN` / `None` |
| 2. Unicode NFKC | Convert compatibility characters to canonical equivalents |
| 3. Lowercase | Case-insensitive token comparison |
| 4. Punctuation → space | Replace `[^\w\s]` with space; **preserves digits/numbers** |
| 5. Whitespace collapse | Collapse multiple spaces; strip leading/trailing whitespace |

**Design principle:** reduce surface noise without discarding meaningful information. Address numbers, non-Latin scripts, and digits are preserved. Aggressive stemming or abbreviation expansion is deferred to the matching stage.

### Before / After Examples

The cells below show raw vs. normalized values for business names and addresses.

In [14]:
import re
import unicodedata

def normalize_text(x):
    if pd.isna(x):
        return ""

    x = str(x)

    # Unicode normalization
    x = unicodedata.normalize("NFKC", x)

    # Lowercase
    x = x.lower()

    # Replace punctuation with spaces
    x = re.sub(r"[^\w\s]", " ", x, flags=re.UNICODE)

    # Normalize whitespace
    x = re.sub(r"\s+", " ", x).strip()

    return x

In [15]:
for df in [s1, s2, s3]:
    df["business_name_norm"] = df["business_name"].apply(normalize_text)
    df["business_address_norm"] = df["business_address"].apply(normalize_text)

In [16]:
display(
    s1[
        [
            "business_name",
            "business_name_norm",
            "business_address",
            "business_address_norm"
        ]
    ].head(10)
)

,business_name,business_name_norm,business_address,business_address_norm
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal
5,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn
8,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...


In [17]:
for name, df in {
    "S1": s1,
    "S2": s2,
    "S3": s3
}.items():

    name_changed = (
        df["business_name"].fillna("").astype(str)
        != df["business_name_norm"]
    ).sum()

    address_changed = (
        df["business_address"].fillna("").astype(str)
        != df["business_address_norm"]
    ).sum()

    print(name)
    print("Name changed:", name_changed, "/", len(df))
    print("Address changed:", address_changed, "/", len(df))
    print()

S1
Name changed: 99098 / 99098
Address changed: 99098 / 99098

S2
Name changed: 106046 / 108019
Address changed: 104406 / 108019

S3
Name changed: 118094 / 120884
Address changed: 116835 / 120884



In [18]:
changed = s1[
    s1["business_name"].fillna("").astype(str)
    != s1["business_name_norm"]
]

display(
    changed[
        [
            "business_name",
            "business_name_norm"
        ]
    ].head(20)
)

,business_name,business_name_norm
0,Orelee's Barbershop,orelee s barbershop
1,Prime Money,prime money
2,B+ Retail Inc,b retail inc
3,Christ Chapel,christ chapel
4,Prabhav Business Center,prabhav business center
5,Custom Wealth Services LLC,custom wealth services llc
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited
7,Nexus Anchor Rain,nexus anchor rain
8,Moore Bitwise Inc,moore bitwise inc
9,Dermatology Green Medicine,dermatology green medicine


In [19]:
changed_addr = s1[
    s1["business_address"].fillna("").astype(str)
    != s1["business_address_norm"]
]

display(
    changed_addr[
        [
            "business_address",
            "business_address_norm"
        ]
    ].head(20)
)

,business_address,business_address_norm
0,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal
5,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue
6,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...
7,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn
8,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in
9,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...


### Exact-Match Coverage After Normalization

After normalization, we check how many S1 entities can be linked to S2/S3 by **exact string equality** on the normalized name or address field. This sets a lower bound on what a simple lookup-based matcher can achieve, and tells us how much of the problem requires fuzzy or semantic matching.

In [20]:
s2_names = set(s2["business_name_norm"].dropna())
s3_names = set(s3["business_name_norm"].dropna())

s1["name_exact_s2"] = s1["business_name_norm"].isin(s2_names)
s1["name_exact_s3"] = s1["business_name_norm"].isin(s3_names)

print("S1 with exact normalized name in S2:",
      s1["name_exact_s2"].sum())

print("S1 with exact normalized name in S3:",
      s1["name_exact_s3"].sum())

S1 with exact normalized name in S2: 6336
S1 with exact normalized name in S3: 7718


In [21]:
s2_addresses = set(s2["business_address_norm"].dropna())
s3_addresses = set(s3["business_address_norm"].dropna())

s1["address_exact_s2"] = s1["business_address_norm"].isin(s2_addresses)
s1["address_exact_s3"] = s1["business_address_norm"].isin(s3_addresses)

print("S1 with exact normalized address in S2:",
      s1["address_exact_s2"].sum())

print("S1 with exact normalized address in S3:",
      s1["address_exact_s3"].sum())

S1 with exact normalized address in S2: 517
S1 with exact normalized address in S3: 228


In [22]:
gt_lookup = gt.set_index("source1_entity_id")["matched_entity_ids"]

s1["ground_truth"] = s1["entity_id"].map(gt_lookup)

display(
    s1[
        [
            "entity_id",
            "business_name",
            "business_name_norm",
            "business_address",
            "business_address_norm",
            "country",
            "ground_truth"
        ]
    ].head(20)
)

,entity_id,business_name,business_name_norm,business_address,business_address_norm,country,ground_truth
0,S1-925783039,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US,NaN
1,S1-773889195,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US,NaN
2,S1-377745466,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US,NaN
3,S1-133037285,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US,NaN
4,S1-755362802,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,India,NaN
5,S1-851869949,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US,NaN
6,S1-785847572,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,India,NaN
7,S1-27541239,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US,NaN
8,S1-629417405,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US,NaN
9,S1-22305073,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,US,NaN


In [ ]:
# GT presence is determined from the S1 ID, not from the match-ID value,
# because zero-match rows have NaN/empty matched_entity_ids and would be
# wrongly excluded by a plain notna() filter.
labeled_s1 = s1[s1["entity_id"].isin(gt_lookup.index)].copy()

def count_matches(x):
    if pd.isna(x) or str(x).strip() == "":
        return 0
    return len(str(x).split(","))

labeled_s1["match_count"] = labeled_s1["ground_truth"].apply(count_matches)

print("Labeled S1 entities:", len(labeled_s1))
print("0 matches :", (labeled_s1["match_count"] == 0).sum())
print("1 match   :", (labeled_s1["match_count"] == 1).sum())
print("Multiple  :", (labeled_s1["match_count"] > 1).sum())

print("\nFull match-count distribution:")
print(labeled_s1["match_count"].value_counts().sort_index())

In [24]:
s2_name = s2.groupby("business_name_norm")["entity_id"].apply(list)
s3_name = s3.groupby("business_name_norm")["entity_id"].apply(list)

def exact_name_candidates(row):
    a = s2_name.get(row["business_name_norm"], [])
    b = s3_name.get(row["business_name_norm"], [])
    return a + b

s1["exact_name_candidates"] = s1.apply(exact_name_candidates, axis=1)

print(
    "S1 with exact normalized name match:",
    (s1["exact_name_candidates"].apply(len) > 0).sum()
)

S1 with exact normalized name match: 11009


In [25]:
s2_addr = s2.groupby("business_address_norm")["entity_id"].apply(list)
s3_addr = s3.groupby("business_address_norm")["entity_id"].apply(list)

def exact_address_candidates(row):
    a = s2_addr.get(row["business_address_norm"], [])
    b = s3_addr.get(row["business_address_norm"], [])
    return a + b

s1["exact_address_candidates"] = s1.apply(
    exact_address_candidates,
    axis=1
)

print(
    "S1 with exact normalized address match:",
    (s1["exact_address_candidates"].apply(len) > 0).sum()
)

S1 with exact normalized address match: 744


In [26]:
def to_set(x):
    if pd.isna(x) or str(x).strip() == "":
        return set()
    return set(str(x).split(","))

s1["gt_set"] = s1["ground_truth"].apply(to_set)

s1["name_set"] = s1["exact_name_candidates"].apply(set)
s1["address_set"] = s1["exact_address_candidates"].apply(set)

s1["name_hit"] = s1.apply(
    lambda r: len(r["gt_set"] & r["name_set"]) > 0,
    axis=1
)

s1["address_hit"] = s1.apply(
    lambda r: len(r["gt_set"] & r["address_set"]) > 0,
    axis=1
)

print("Exact name candidates:", (s1["name_set"].apply(len) > 0).sum())
print("Correct name candidates:", s1["name_hit"].sum())

print()

print("Exact address candidates:", (s1["address_set"].apply(len) > 0).sum())
print("Correct address candidates:", s1["address_hit"].sum())

Exact name candidates: 11009
Correct name candidates: 58

Exact address candidates: 744
Correct address candidates: 24


In [ ]:
name_candidates = (s1["name_set"].apply(len) > 0).sum()
name_correct = s1["name_hit"].sum()

address_candidates = (s1["address_set"].apply(len) > 0).sum()
address_correct = s1["address_hit"].sum()

print(f"Name exact-match overlap (S1 entities with an exact-name candidate that share a GT match): {name_correct / name_candidates * 100:.2f}%")
print(f"Address exact-match overlap (S1 entities with an exact-address candidate that share a GT match): {address_correct / address_candidates * 100:.2f}%")

In [28]:
display(
    s1[
        s1["name_hit"]
    ][
        [
            "entity_id",
            "business_name",
            "business_name_norm",
            "country",
            "ground_truth",
            "exact_name_candidates"
        ]
    ].head(20)
)

,entity_id,business_name,business_name_norm,country,ground_truth,exact_name_candidates
227,S1-259452306,Shorter and Reyes Xrp,shorter and reyes xrp,US,"S2-547115644,S3-787015231,S3-118331403",[S3-787015231]
968,S1-401074421,Chau Minerals LLC,chau minerals llc,US,"S3-597319704,S3-303073926,S3-881527814",[S3-303073926]
3604,S1-164676607,"Farquhar Arrow of Altamont, LLC",farquhar arrow of altamont llc,US,"S2-871065462,S2-265752783,S3-312468298",[S3-312468298]
3930,S1-579682598,Fine Technologies Private Limited,fine technologies private limited,India,"S2-163820222,S2-381247126,S3-824276728",[S3-824276728]
6029,S1-660621852,Granites Extrusion Private Limited,granites extrusion private limited,India,"S2-623142135,S2-413292907,S3-557379316,S3-6349...",[S2-623142135]
6471,S1-623087726,The Dent Yoga,the dent yoga,US,"S2-315128643,S2-869829982,S2-156629748,S3-6701...","[S2-219352693, S2-156629748, S3-250475047]"
6516,S1-852429384,Colonial Partners Inc,colonial partners inc,US,"S2-279935900,S2-667139861,S3-380319648,S3-5642...",[S3-380319648]
7746,S1-94578048,Julianna Ortiz Best Metal LLC,julianna ortiz best metal llc,US,"S2-675323999,S2-229705326,S2-107845340,S3-8620...",[S3-879415165]
7767,S1-210849781,Urology Partners Inc,urology partners inc,US,"S2-562117280,S2-845708664,S2-820221567,S3-6499...","[S2-568772906, S2-845708664, S2-283750653, S3-..."
10582,S1-187690902,Housing Charities,housing charities,US,"S2-196753696,S2-960373594,S3-188063360,S3-4116...","[S3-188063360, S3-625032608]"


---

## 10. Key Findings

*(Based on the actual outputs observed above.)*

**Dataset sizes**
- Source 1 contains 99,098 entities, Source 2 contains 108,019, and Source 3 contains 120,884.
- The ground-truth file provides labels for 72,960 Source 1 entities (the remaining ~26 k S1 entities have no GT row).

**Data quality**
- Business names are complete across all three sources.
- Source 2 and Source 3 contain missing/empty business addresses.
- Country has one missing value in each source.
- Entity IDs are unique within each source.

**Country observations**
- Training data contains US and India records.
- Country is treated as an open-set string and is not hard-coded to a fixed list.

**Ground-truth match distribution**
- Among the 72,960 labeled S1 entities, 4,090 have zero matches, 3,921 have exactly one match, and 64,949 have multiple matches.
- One-to-many matching is therefore the dominant case and a central part of the problem.

**Name and address noise patterns**
- Punctuation differences (periods, commas, hyphens) are the most common noise source.
- Capitalization and abbreviation variations are widespread.
- Some records contain Unicode compatibility variations (e.g., fullwidth forms, combining characters) that NFKC normalization resolves; it does not transliterate accented characters.

**Normalization effectiveness**
- Normalization changes a significant proportion of names and addresses across all sources.
- Exact normalized name/address equality identifies only a subset of possible matches, motivating fuzzy similarity and learned pairwise matching features.
- The exact-match coverage figures in Section 9 show how many S1 entities with an exact-name candidate also share a known GT match; this is *not* candidate recall — proper candidate-recall evaluation will be done in M4.


---

## 11. Handoff to Matching Team

This notebook delivers:

| Artifact | Description |
|---|---|
| `business_name_norm` column | Normalized business name for each source (added in-place) |
| `business_address_norm` column | Normalized business address for each source |
| `src/preprocess.py` | Reusable `normalize_text()`, `normalize_name()`, `normalize_address()` functions |
| `docs/dataset_dictionary.md` | Full field-level schema documentation |

**Out of scope for this notebook:**
- Fuzzy/semantic matching
- Blocking and candidate generation
- Feature engineering
- Model training, threshold tuning, and final submission

Those stages are handled by downstream team members using the normalized representations and EDA findings produced here.